# UN Comtrade API — Test Notebook (Comores)

This notebook uses the [`comtradeapicall`](https://github.com/uncomtrade/comtradeapicall) Python library to:
1. Check if Comoros (reporterCode **174**) has trade data beyond 2021
2. Demonstrate the main API functions (free preview + subscription-based)

**Key constants:**
- Comoros M49 code: `174`
- Free preview: up to **500 rows**, no key needed → `previewFinalData()`
- Full data: up to **250 000 rows**, free subscription key needed → `getFinalData()`

## 1. Install and Import Required Libraries

In [ ]:
!pip install comtradeapicall pandas -q

import comtradeapicall
import pandas as pd

# Verify version
import importlib.metadata
print("comtradeapicall version:", importlib.metadata.version("comtradeapicall"))
print("pandas version:", pd.__version__)

## 2. Set Up API Subscription Key

Set your free subscription key from [comtradedeveloper.un.org](https://comtradedeveloper.un.org).  
Leave empty (`''`) to use only the free preview endpoints (max 500 rows).

In [ ]:
# Paste your free key here (from comtradedeveloper.un.org → Profile → Show keys)
subscription_key = ''   # leave empty to use preview endpoints only

# Output directory for bulk downloads
import os
directory = os.path.join(os.path.dirname(os.getcwd()), 'outputs', 'comtrade_bulk')
os.makedirs(directory, exist_ok=True)

# Comoros constants
COMOROS_CODE = 174       # M49 numeric code for Union des Comores
COMOROS_ISO3 = 'COM'

print(f"Subscription key set: {'YES' if subscription_key else 'NO (preview only)'}")
print(f"Output directory: {directory}")

## 3. Preview Final Data — Free, No Key (max 500 rows)

Uses `previewFinalData()`. This is the **first thing to run** to check whether Comoros has data beyond 2021.  
Parameters used here:
- `typeCode='C'` — commodities
- `freqCode='A'` — annual
- `clCode='HS'` — Harmonized System
- `period` — years to test
- `reporterCode='174'` — Comoros (M49)
- `flowCode='M,X'` — both imports and exports
- `cmdCode='TOTAL'` — all products combined

In [ ]:
# --- 3a. Check data availability for 2021 (should exist via WITS) ---
df_2021 = comtradeapicall.previewFinalData(
    typeCode='C', freqCode='A', clCode='HS',
    period='2021',
    reporterCode=str(COMOROS_CODE),
    cmdCode='TOTAL',
    flowCode='M',
    partnerCode=None, partner2Code=None,
    customsCode=None, motCode=None,
    maxRecords=500, format_output='JSON',
    aggregateBy=None, breakdownMode='classic',
    countOnly=None, includeDesc=True
)
print(f"2021 rows returned: {len(df_2021) if df_2021 is not None else 0}")
if df_2021 is not None and len(df_2021) > 0:
    print(df_2021[['period', 'reporterDesc', 'flowDesc', 'primaryValue']].head())

In [ ]:
# --- 3b. KEY TEST: Check if Comoros has data for 2022, 2023, 2024 ---
for year in ['2022', '2023', '2024']:
    df = comtradeapicall.previewFinalData(
        typeCode='C', freqCode='A', clCode='HS',
        period=year,
        reporterCode=str(COMOROS_CODE),
        cmdCode='TOTAL',
        flowCode='M',
        partnerCode=None, partner2Code=None,
        customsCode=None, motCode=None,
        maxRecords=500, format_output='JSON',
        aggregateBy=None, breakdownMode='classic',
        countOnly=None, includeDesc=True
    )
    n = len(df) if df is not None else 0
    status = "✅ DATA FOUND" if n > 0 else "❌ No data"
    print(f"Year {year}: {status} ({n} rows)")

## 4. Preview Tariff Line Data (free, no key, max 500 rows)

`previewTarifflineData()` returns HS-6 level detail (individual commodity codes like vanilla = 090500).

In [ ]:
# Preview tariff line data for Comoros exports in 2021 (HS6 level)
# Vanilla = 090500, Cloves = 090700
df_tariff = comtradeapicall.previewTarifflineData(
    typeCode='C', freqCode='A', clCode='HS',
    period='2021',
    reporterCode=str(COMOROS_CODE),
    cmdCode='0905,0907',   # vanilla and cloves at HS6
    flowCode='X',
    partnerCode=None, partner2Code=None,
    customsCode=None, motCode=None,
    maxRecords=500, format_output='JSON',
    countOnly=None, includeDesc=True
)
print(f"Tariff line rows: {len(df_tariff) if df_tariff is not None else 0}")
if df_tariff is not None and len(df_tariff) > 0:
    df_tariff[['period', 'cmdCode', 'cmdDesc', 'partnerDesc', 'primaryValue']].head(10)

## 5. Preview HS Commodity Codes

List the HS commodity codes available in the Comtrade system (useful to find exact codes like vanilla, cloves, ylang-ylang).

In [ ]:
# Get reference list — search for vanilla and cloves
df_cmd = comtradeapicall.getReference('cmd')
# Filter to show spice-related codes (HS chapter 09 = coffee, tea, spices)
mask = df_cmd['id'].astype(str).str.startswith('09')
spices = df_cmd[mask][['id', 'text']].copy()
print(f"Total HS codes in chapter 09: {len(spices)}")
spices.head(20)

## 6. Reporter and Partner Areas

List all available reporter (and partner) country codes — useful to confirm that `174` is the correct code for Comoros.

In [ ]:
# Get reporter areas and find Comoros
df_reporters = comtradeapicall.getReference('reporter')
comoros_row = df_reporters[df_reporters['text'].str.contains('Comoro', case=False, na=False)]
print("Comoros in reporter list:")
print(comoros_row[['id', 'text']].to_string())

# Also check partner list
df_partners = comtradeapicall.getReference('partner')
comoros_partner = df_partners[df_partners['text'].str.contains('Comoro', case=False, na=False)]
print("\nComoros in partner list:")
print(comoros_partner[['id', 'text']].to_string())

## 7. Full Data with Subscription Key — `getFinalData()`

> **Requires** a free subscription key from [comtradedeveloper.un.org](https://comtradedeveloper.un.org).  
> Returns up to **250 000 rows** per call (vs 500 for preview).

This is the function to use to pull the full Comoros dataset for 2022–2024 if data is available.

In [ ]:
if not subscription_key:
    print("⚠️  No subscription key set. Skipping getFinalData() call.")
    print("   Register at comtradedeveloper.un.org → Products → Free APIs → Subscribe")
else:
    # Pull full Comoros annual data (imports + exports) for 2022, 2023, 2024
    df_full = comtradeapicall.getFinalData(
        subscription_key,
        typeCode='C', freqCode='A', clCode='HS',
        period='2022,2023,2024',
        reporterCode=str(COMOROS_CODE),
        cmdCode='AG2',         # HS 2-digit aggregates (section level)
        flowCode='M,X',
        partnerCode=None, partner2Code=None,
        customsCode=None, motCode=None,
        maxRecords=250000, format_output='JSON',
        aggregateBy=None, breakdownMode='classic',
        countOnly=None, includeDesc=True
    )
    if df_full is not None and len(df_full) > 0:
        print(f"✅ {len(df_full)} rows returned for 2022–2024")
        print(df_full[['period', 'flowDesc', 'cmdDesc', 'primaryValue']].head(10))
    else:
        print("❌ No data returned for those years (Comoros may not have submitted yet)")

## 8. Bulk Download Final Data

> **Requires** a subscription key (free or paid).

Downloads complete annual files for Comoros to the local `directory`. Each file is a gzip CSV.

In [ ]:
if not subscription_key:
    print("⚠️  No subscription key — skipping bulk download.")
else:
    # Download all annual HS data for Comoros (all years available)
    comtradeapicall.bulkDownloadFinalFile(
        subscription_key,
        directory,
        typeCode='C',
        freqCode='A',
        clCode='HS',
        period=None,           # None = all available years
        reporterCode=COMOROS_CODE,
        decompress=True        # decompress .gz files automatically
    )
    print(f"Files saved to: {directory}")

## 9. Check Data Availability Before Downloading

Use `getFinalDataAvailability()` to see exactly which years/periods Comoros data is available — before committing to a download.

In [ ]:
if not subscription_key:
    print("⚠️  No key — skipping availability check via authenticated endpoint.")
    print("   Run cell 3b (previewFinalData) above to check without a key.")
else:
    # Check which years of annual HS data are available for Comoros
    df_avail = comtradeapicall.getFinalDataAvailability(
        subscription_key,
        typeCode='C',
        freqCode='A',
        clCode='HS',
        period=None,           # all periods
        reporterCode=COMOROS_CODE
    )
    if df_avail is not None and len(df_avail) > 0:
        print(f"✅ {len(df_avail)} dataset(s) available for Comoros")
        cols = ['period', 'reporterISO', 'reporterDesc', 'totalRecords', 'lastReleased']
        print(df_avail[cols].sort_values('period').to_string(index=False))
    else:
        print("❌ No availability data returned for Comoros")